# 01 — Data Exploration

Produces dataset statistics and figures consumed by `website/slides.qmd`.

Outputs written to `figures/datasets/`.

Run with: `uv run jupyter nbconvert --to notebook --execute notebooks/01_data_exploration.ipynb`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

FIGURES_DIR = Path('../figures/datasets')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Figures → ', FIGURES_DIR.resolve())

In [ ]:
# Load all four manifests
PROC = Path('../data/processed')

manifests = {
    'Severstal':     pd.read_parquet(PROC / 'severstal_manifest.parquet'),
    'NEU-DET':       pd.read_parquet(PROC / 'neu_det_manifest.parquet'),
    'KolektorSDD2':  pd.read_parquet(PROC / 'kolektor_manifest.parquet'),
    'GC10-DET':      pd.read_parquet(PROC / 'gc10_manifest.parquet'),
}

for name, df in manifests.items():
    print(f'{name:15s}: {len(df):,} images  '
          f'{df["has_defect"].sum():,} defect ({100*df["has_defect"].mean():.0f}%)  '
          f'splits: {dict(df["split"].value_counts())}')

In [ ]:
# --- Class distribution per dataset -------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

colors = ['#1f77b4', '#ff7f0e']
for ax, (name, df) in zip(axes, manifests.items()):
    counts = [int((~df['has_defect']).sum()), int(df['has_defect'].sum())]
    bars = ax.bar(['Normal', 'Defect'], counts, color=colors)
    ax.set_title(name, fontsize=11)
    ax.set_ylabel('Image count')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{cnt:,}', ha='center', va='bottom', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Dataset class balance', fontsize=13)
fig.tight_layout()
out = FIGURES_DIR / 'class_distribution.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'→ {out}')

In [ ]:
# --- Train/val/test split breakdown -------------------------------------------
split_colors = {'train': '#2ca02c', 'val': '#9467bd', 'test': '#d62728'}
split_order  = ['train', 'val', 'test']

dataset_names = list(manifests.keys())
x = np.arange(len(dataset_names))

fig, ax = plt.subplots(figsize=(9, 5))
bottoms = np.zeros(len(dataset_names))

for split in split_order:
    counts = [manifests[n]['split'].eq(split).sum() for n in dataset_names]
    ax.bar(x, counts, bottom=bottoms, color=split_colors[split], label=split, width=0.6)
    for xi, (c, b) in enumerate(zip(counts, bottoms)):
        if c > 0:
            ax.text(xi, b + c / 2, f'{c:,}', ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    bottoms += np.array(counts)

ax.set(title='Train / Val / Test split per dataset', ylabel='Image count',
       xticks=x, xticklabels=dataset_names)
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / 'split_breakdown.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'→ {out}')

In [ ]:
# --- Sample image grid (2 images per dataset × defect/normal) -----------------
rows, cols = 4, 4   # dataset × (2 defect + 2 normal)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 2.8))

rng = np.random.default_rng(42)

for row_idx, (ds_name, df) in enumerate(manifests.items()):
    defects = df[df['has_defect']].sample(min(2, df['has_defect'].sum()), random_state=42)
    normals = df[~df['has_defect']].sample(min(2, (~df['has_defect']).sum()), random_state=42)
    samples = list(defects.itertuples()) + list(normals.itertuples())

    for col_idx, row in enumerate(samples):
        ax = axes[row_idx][col_idx]
        try:
            img = Image.open(row.path).convert('RGB')
            img.thumbnail((300, 300))
            ax.imshow(img)
        except Exception:
            ax.set_facecolor('#eee')
        label = 'DEFECT' if row.has_defect else 'NORMAL'
        color = '#d62728' if row.has_defect else '#1f77b4'
        ax.set_title(f'{ds_name}\n{label}', fontsize=8, color=color)
        ax.axis('off')

fig.suptitle('Sample images across datasets', fontsize=13, y=1.01)
fig.tight_layout()
out = FIGURES_DIR / 'sample_grid.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
plt.close(fig)
print(f'→ {out}')

In [ ]:
# --- Defect class distribution (NEU-DET and GC10 have class labels) -----------
for ds_name, col in [('NEU-DET', 'defect_class'), ('GC10-DET', 'defect_class')]:
    df = manifests[ds_name]
    if col not in df.columns:
        print(f'{ds_name}: no {col!r} column — skip')
        continue
    counts = df[df['has_defect']][col].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.sort_values().plot.barh(ax=ax, color='#2ca02c')
    ax.set(title=f'{ds_name} — defect class distribution',
           xlabel='Image count')
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    out = FIGURES_DIR / f'class_dist_{ds_name.lower().replace("-","_")}.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'→ {out}')

In [ ]:
# --- Image dimension summary --------------------------------------------------
print(f'\n{"Dataset":<15} {"Col (width)">12} {"Row (height)">13} {"Aspect">8}')
print('-' * 52)

for ds_name, df in manifests.items():
    sample = df.sample(min(20, len(df)), random_state=0)
    widths, heights = [], []
    for p in sample['path']:
        try:
            w, h = Image.open(p).size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass
    if widths:
        w_med = int(np.median(widths))
        h_med = int(np.median(heights))
        print(f'{ds_name:<15} {w_med:>12} {h_med:>13} {w_med/h_med:>8.2f}')

In [ ]:
# --- Summary table for slides -------------------------------------------------
rows_list = []
for ds_name, df in manifests.items():
    rows_list.append({
        'Dataset':   ds_name,
        'Total':     len(df),
        'Defect':    int(df['has_defect'].sum()),
        'Normal':    int((~df['has_defect']).sum()),
        'Defect%':   f"{100*df['has_defect'].mean():.0f}%",
        'Role':      'Train+Val' if ds_name in ('Severstal', 'NEU-DET') else 'Held-out test',
    })
summary_df = pd.DataFrame(rows_list)
print(summary_df.to_markdown(index=False))
summary_df.to_csv(FIGURES_DIR / 'dataset_summary.csv', index=False)
print(f'\n→ {FIGURES_DIR / "dataset_summary.csv"}')